# 02 · Profile: NYS DOL long-term occupational projections (4 NYC editions)

**Method:** `docs/framework.md` §3 (the 10-section exploration template). Card: `docs/source-cards/nysdol_projections.md`.

**Why this source matters.** It is the only source that gives projected NYC employment and annual openings by job. That makes it the demand side of brief Q1 (*which jobs are growing or shrinking, and how many openings are there each year?*), and the number graduates get compared against.

**What this notebook does:** reads the four raw files, checks grain, codes, coverage and internal arithmetic, and ends with an issues list and a proposed staging table. It changes nothing. Treatments are decided in staging.

Every read uses `dtype=str, keep_default_na=False`, so nothing is coerced before we've looked at it.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NUM = ["Base Year", "Projected Year", "Change", "Percent",
       "Total Annual Openings", "Growth Annual Openings", "Replacement Annual Openings"]

## 0. Source header

| | |
|---|---|
| **Publisher** | New York State Department of Labor (NYS DOL), Division of Research & Statistics |
| **Dataset** | *Long-Term Occupational Employment Projections*, data.ny.gov `pqm4-9qqb` |
| **How it was collected** | **A model, not a survey.** Four phases ([technical notes](https://dol.ny.gov/long-term-occupational-projections-technical-notes)): (1) project industry employment; (2) build industry staffing patterns from the OEWS survey (~57,000 NY employers); (3) apply the patterns to get occupational employment, base and projected; (4) add growth to separations to get openings |
| **What "employment" counts** | **Jobs, not people**, located where the employer is (OEWS concept). The technical notes don't say whether self-employed workers are included |
| **Openings** | Openings = change + separations (labor-force exits + occupational transfers). Job-to-job moves within the same occupation aren't counted. This is BLS's separations method, first used for the 2016–26 round ([BLS](https://www.bls.gov/emp/documentation/separations.htm)) |
| **Unit of time** | Annual openings are an average over the 10 years, not a forecast for any one year (technical notes) |
| **Stated limits** | Doesn't model immigration, new occupations, relocation of jobs, or shocks such as recessions (technical notes) |
| **Editions** | Every 2 years. The portal keeps only the latest, so older editions come from Wayback captures (see manifest) |
| **Geography** | NY State plus 10 labor-market regions. "New York City" = the 5 boroughs as one region. Nothing smaller is published |

In [2]:
man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
FILES = man[man.key.isin(["nysdol_projections", "nysdol_projections_archive"])][["file", "url", "downloaded_utc", "sha256", "note"]]
FILES = FILES.sort_values("note").reset_index(drop=True)
FILES[["file", "downloaded_utc", "note"]]

,file,downloaded_utc,note
0,jobs/nysdol_projections/pqm4-9qqb_2026-10-12.csv,2026-10-12T17:03:42+00:00,"Long-term projections, current vintage (NYC 2022-2032)"
1,jobs/nysdol_projections_archive/pqm4-9qqb_2016-2026_wayback2021032...,2026-10-12T17:03:46+00:00,"NYC 2016-2026 vintage, Wayback capture 20210324183534"
2,jobs/nysdol_projections_archive/pqm4-9qqb_2018-2028_wayback2022113...,2026-10-12T17:03:48+00:00,"NYC 2018-2028 vintage, Wayback capture 20221130141045"
3,jobs/nysdol_projections_archive/pqm4-9qqb_2020-2030_wayback2024110...,2026-10-12T17:03:53+00:00,"NYC 2020-2030 vintage, Wayback capture 20241104204940"


## 1. Load as text

All four files share one 11-column layout. A leading apostrophe on SOC codes (an Excel text marker) is stripped only for the profiling below. That also goes on the issues list.

In [3]:
raw = {}
for f in FILES.file:
    d = pd.read_csv(RAW / f, dtype=str, keep_default_na=False)
    raw[f] = d
    print(f"{f:<75} {d.shape[0]:>5} rows x {d.shape[1]} cols   apostrophe SOCs: {d.SOC.str.startswith(chr(39)).sum()}")

cols = {tuple(d.columns) for d in raw.values()}
print("\nIdentical column layout across files:", len(cols) == 1)
list(cols)[0]

jobs/nysdol_projections/pqm4-9qqb_2026-10-12.csv                             6758 rows x 11 cols   apostrophe SOCs: 0
jobs/nysdol_projections_archive/pqm4-9qqb_2016-2026_wayback20210324183534.csv  6626 rows x 11 cols   apostrophe SOCs: 6626
jobs/nysdol_projections_archive/pqm4-9qqb_2018-2028_wayback20221130141045.csv  6617 rows x 11 cols   apostrophe SOCs: 6617
jobs/nysdol_projections_archive/pqm4-9qqb_2020-2030_wayback20241104204940.csv  6596 rows x 11 cols   apostrophe SOCs: 0

Identical column layout across files: True


('Area',
 'Period',
 'SOC',
 'Occupation Title',
 'Base Year',
 'Projected Year',
 'Change',
 'Percent',
 'Total Annual Openings',
 'Growth Annual Openings',
 'Replacement Annual Openings')

## 2. What one row is (grain)

**Presumed key:** `Area × Period × SOC`.

The first look in notebook 01 flagged a trap here: each file mixes two editions. The regions (NYC included) are one edition, while the statewide rows are the next one, two years newer. So "one file = one edition" is false. The edition has to come from `Period` on each row, never from the file name.

In [4]:
grain = []
for f, d in raw.items():
    d = d.assign(SOC=d.SOC.str.lstrip("'"))
    for (a_is_nyc, period), g in d.groupby([d.Area.eq("New York City"), "Period"]):
        grain.append({"file": f.split("/")[-1][:35], "slice": "NYC" if a_is_nyc else "statewide / other regions",
                      "Period": period, "rows": len(g)})
display(pd.DataFrame(grain).pivot_table(index="file", columns="slice", values="Period", aggfunc=lambda s: " | ".join(sorted(set(s)))))

for f, d in raw.items():
    d = d.assign(SOC=d.SOC.str.lstrip("'"))
    k = d.duplicated(["Area", "Period", "SOC"], keep=False)
    print(f"{f.split('/')[-1][:45]:<46} duplicate keys: {k.sum():>3}   of which exact duplicate rows: {d[k].duplicated(keep=False).sum()}")

slice,NYC,statewide / other regions
file,,
pqm4-9qqb_2016-2026_wayback20210324,2016 - 2026,2016 - 2026 | 2018 - 2028
pqm4-9qqb_2018-2028_wayback20221130,2018 - 2028,2018 - 2028 | 2020 - 2030
pqm4-9qqb_2020-2030_wayback20241104,2020-2030,2020-2030 | 2022 - 2032
pqm4-9qqb_2026-10-12.csv,2022 - 2032,2022 - 2032 | 2024 - 2034


pqm4-9qqb_2026-10-12.csv                       duplicate keys:  10   of which exact duplicate rows: 0
pqm4-9qqb_2016-2026_wayback20210324183534.csv  duplicate keys:   0   of which exact duplicate rows: 0
pqm4-9qqb_2018-2028_wayback20221130141045.csv  duplicate keys:  12   of which exact duplicate rows: 12
pqm4-9qqb_2020-2030_wayback20241104204940.csv  duplicate keys: 119   of which exact duplicate rows: 63


In [5]:
# What the duplicates are
for f, d in raw.items():
    d = d.assign(SOC=d.SOC.str.lstrip("'"))
    k = d.duplicated(["Area", "Period", "SOC"], keep=False)
    if k.any():
        print(f)
        display(d[k].groupby(["Area", "SOC", "Occupation Title"]).size().to_frame("copies").head(10))

jobs/nysdol_projections/pqm4-9qqb_2026-10-12.csv


copies
Area               SOC     Occupation Title                                              
New York Statewide 53-7062 Laborers and Freight, Stock, and Material Movers, Hand       2
                   53-7063 Machine Feeders and Offbearers                               2
                   53-7064 Packers and Packagers, Hand                                  2
                   53-7065 Stockers and Order Fillers                                   2
                   53-7081 Refuse and Recyclable Material Collectors                    2

jobs/nysdol_projections_archive/pqm4-9qqb_2018-2028_wayback20221130141045.csv


,,,copies
Area,SOC,Occupation Title,
Capital Region,39-7000,Tour and Travel Guides,2
Central Region,39-7000,Tour and Travel Guides,2
Finger Lakes,39-7000,Tour and Travel Guides,2
Hudson Valley,39-7000,Tour and Travel Guides,2
New York City,39-7000,Tour and Travel Guides,2
Western Region,39-7000,Tour and Travel Guides,2


jobs/nysdol_projections_archive/pqm4-9qqb_2020-2030_wayback20241104204940.csv


copies
Area           SOC     Occupation Title                                                 
Central Region                                                                        13
               51-9161 Computer Numerically Controlled Tool Operators                  2
               51-9191 Adhesive Bonding Machine Operators and Tenders                  2
               51-9195 Molders, Shapers, and Casters, Except Metal and Plastic         2
               51-9196 Paper Goods Machine Setters, Operators, and Tenders             2
               51-9198 Helpers--Production Workers                                     2
               51-9199 Production Workers, All Other                                   2
               53-0000 Transportation and Material Moving Occupations                  2
               53-1000 Supervisors of Transportation and Material Moving Workers       2
               53-2000 Air Transportation Workers                                      2

**Grain findings**
- Duplicates exist, but none affect NYC's numbers. (a) 2018-28: `39-7000 Tour and Travel Guides` is repeated as an exact duplicate row in 6 regions, NYC included. (b) 2020-30: 63 fully blank rows (Central Region), plus Central Region codes such as `51-9161` that appear twice with different values. (c) 2022-32: 5 statewide codes (`53-706x`, `53-7081`) appear twice with different values. (b) and (c) are conflicting duplicates outside NYC. They're out of scope, but they're a warning about how this portal extract is assembled.
- **NYC slice:** after dropping the one exact duplicate, the key is unique in every edition. The NYC grain is one row = one SOC code × one edition, with codes from every level of the hierarchy (§4).

From here on, the analysis works on the NYC slice of each file, which is the one edition per file we care about.

In [6]:
EDITION = {}
nyc = []
for f, d in raw.items():
    d = d.assign(SOC=d.SOC.str.lstrip("'"))
    d = d[d.Area == "New York City"].drop_duplicates()
    ed = d.Period.str.replace(" ", "").iloc[0]
    assert d.Period.nunique() == 1 and not d.duplicated(["SOC"]).any()
    EDITION[f] = ed
    nyc.append(d.assign(edition=ed, file=f))
nyc = pd.concat(nyc, ignore_index=True)
for c in NUM:
    nyc[c + "_n"] = pd.to_numeric(nyc[c])   # raises if anything non-numeric -> that is a check in itself
nyc = nyc.sort_values(["edition", "SOC"]).reset_index(drop=True)
nyc.groupby("edition").agg(rows=("SOC", "size"), base_year_employment_total=("Base Year_n", "max"))

,rows,base_year_employment_total
edition,,
2016-2026,655,4684200
2018-2028,637,4939450
2020-2030,626,4566620
2022-2032,657,4850820


## 3. Column profile (NYC slice)

`Base Year` and `Projected Year` are misleading names: they hold employment counts, not years. The years only appear inside `Period`.

In [7]:
prof = []
for c in raw[FILES.file[0]].columns:
    s = nyc[c]
    prof.append({"column": c, "blank": (s == "").sum(), "distinct": s.nunique(),
                 "pattern(top)": s.str.replace(r"\d", "9", regex=True).value_counts().index[0],
                 "examples": ", ".join(s.drop_duplicates().head(3))})
display(pd.DataFrame(prof))
nyc.groupby("edition")[[c + "_n" for c in NUM]].describe().T.loc[(slice(None), ["min", "50%", "max"]), :].round(1)

,column,blank,distinct,pattern(top),examples
0,Area,0,1,New York City,New York City
1,Period,0,4,9999 - 9999,"2016 - 2026, 2018 - 2028, 2020-2030"
2,SOC,0,801,99-9999,"00-0000, 11-0000, 11-1000"
3,Occupation Title,0,808,Tour and Travel Guides,"Total, All Occupations, Management Occupations, Top Executives"
4,Base Year,0,1357,9999,"4684200, 294230, 92210"
5,Projected Year,0,1446,9999,"5325230, 335350, 104260"
6,Change,0,767,999,"641030, 41120, 12050"
7,Percent,0,1074,99.9,"13.7, 14, 13.1"
8,Total Annual Openings,0,1395,999,"593100, 28460, 9000"
9,Growth Annual Openings,0,743,99,"64100, 4110, 1210"


edition                            2016-2026  2018-2028  2020-2030  2022-2032
Base Year_n                   min      260.0      250.0      260.0      250.0
                              50%     3930.0     4200.0     4005.0     3410.0
                              max  4684200.0  4939450.0  4566620.0  4850820.0
Projected Year_n              min      250.0      260.0      260.0      250.0
                              50%     4330.0     4540.0     5065.0     3620.0
                              max  5325230.0  5543240.0  5704650.0  5435330.0
Change_n                      min    -8380.0   -23480.0    -5910.0   -16460.0
                              50%      420.0      310.0      755.0      250.0
                              max   641030.0   603790.0  1138030.0   584510.0
Percent_n                     min      -29.1      -31.8      -29.1      -28.9
                              50%       12.5        9.3       19.4       11.3
                              max       55.0       71.9      135.8       61.3
Total Annual Openings_n       min       20.0       21.0       21.0        1.0
                              50%      430.0      482.0      503.0      352.0
                              max   593100.0   641095.0   653913.0   605681.0
Growth Annual Openings_n      min     -840.0    -2348.0     -591.0    -1646.0
                              50%       40.0       31.0       75.5       25.0
                              max    64100.0    60379.0   113803.0    58451.0
Replacement Annual Openings_n min       10.0       20.0       20.0        0.0
                              50%      380.0      440.0      430.0      320.0
                              max   529000.0   580720.0   540110.0   547230.0

**Profile findings**
- No blanks and no suppression markers in the NYC slice. Every numeric cell parses as a number. Suppression shows up as missing rows instead (§7).
- Employment is rounded to 10s everywhere, and the smallest NYC row is 250 jobs. That looks like a publication floor, but the docs don't confirm it.
- `Percent` has 1 decimal in the older editions and 2 in 2022-32. That's cosmetic.
- **Negative `Change`** (declining jobs) carries negative `Growth Annual Openings` in the separations-method files. Openings net out decline, so `Total = Growth + Replacement` can be smaller than replacement alone.

## 4. Codes, hierarchy and special values

SOC codes come in four levels, read from the digits (SOC 2018 structure):

| Level | Pattern | Example |
|---|---|---|
| major | `XX-0000` | 15-0000 Computer and Mathematical |
| minor | `XX-XX00` | 15-1200 Computer Occupations |
| broad | `XX-XXX0` | 31-1120 Home Health and Personal Care Aides |
| detailed | `XX-XXXX` | 15-1252 Software Developers |

`00-0000` is the all-occupations total. The file stacks all levels, so summing a column naively counts every job 3–4 times. A row is a leaf when no lower-level code below it appears in the same edition. Leaves are the only rows that can be summed.

In [8]:
def soc_level(s: str) -> str:
    if s == "00-0000": return "total"
    t = s[3:]
    return "major" if t == "0000" else "minor" if t.endswith("00") else "broad" if t.endswith("0") else "detailed"

PREFIX = {"total": 0, "major": 2, "minor": 4, "broad": 5, "detailed": 6}   # digits that define the group (dash removed)
RANK = {"total": 0, "major": 1, "minor": 2, "broad": 3, "detailed": 4}

nyc["level"] = nyc.SOC.map(soc_level)
def is_leaf(g):
    codes = [(c.replace("-", ""), soc_level(c)) for c in g.SOC]
    return g.apply(lambda r: not any(c.startswith(r.SOC.replace("-", "")[:PREFIX[r.level]]) and RANK[l] > RANK[r.level]
                                     for c, l in codes), axis=1)
nyc["leaf"] = nyc.groupby("edition", group_keys=False)[["SOC", "level"]].apply(is_leaf)

display(pd.crosstab(nyc.edition, [nyc.level], margins=True))
print("Leaves by level (rows that can be summed):")
pd.crosstab(nyc.edition, nyc[nyc.leaf].level)

level,broad,detailed,major,minor,total,All
edition,,,,,,
2016-2026,0,542,22,90,1,655
2018-2028,3,523,20,90,1,637
2020-2030,10,499,22,94,1,626
2022-2032,7,533,22,94,1,657
All,20,2097,86,368,4,2575


Leaves by level (rows that can be summed):


level,broad,detailed,minor
edition,,,
2016-2026,0,542,4
2018-2028,3,523,6
2020-2030,10,499,3
2022-2032,7,533,3


**The hierarchy is ragged.** Most leaves are detailed codes, but in some places NYS DOL publishes only a broad or minor group with no detail below it. For example, `31-1120 Home Health and Personal Care Aides` is ~389k jobs in 2022-32 and isn't split into home health aides and personal care aides. So in staging, "leaf" is a flag computed per edition, not a level.

**Which SOC edition?** Codes that exist only in SOC 2010 vs only in SOC 2018 tell the editions apart:

In [9]:
xw = pd.read_excel(RAW / "jobs/bls_soc_crosswalk/soc_2010_to_2018_crosswalk.xlsx", header=8, dtype=str)
xw.columns = ["soc2010", "title2010", "soc2018", "title2018"]
xw = xw.dropna(subset=["soc2010"])
only2010 = set(xw.soc2010) - set(xw.soc2018)
only2018 = set(xw.soc2018) - set(xw.soc2010)

det = nyc[nyc.level == "detailed"]
det.groupby("edition").agg(detailed=("SOC", "size"),
                          in_2010_only=("SOC", lambda s: s.isin(only2010).sum()),
                          in_2018_only=("SOC", lambda s: s.isin(only2018).sum()),
                          in_neither=("SOC", lambda s: (~s.isin(set(xw.soc2010) | set(xw.soc2018))).sum()))

,detailed,in_2010_only,in_2018_only,in_neither
edition,,,,
2016-2026,542,60,0,2
2018-2028,523,57,0,6
2020-2030,499,2,35,22
2022-2032,533,2,38,14


In [10]:
# The codes in neither list: NYS DOL / BLS projection aggregates (e.g. 15-1256 = 15-1252 + 15-1253)
neither = det[~det.SOC.isin(set(xw.soc2010) | set(xw.soc2018))]
neither.pivot_table(index=["SOC", "Occupation Title"], columns="edition", values="Base Year_n", aggfunc="first")

,edition,2016-2026,2018-2028,2020-2030,2022-2032
SOC,Occupation Title,,,,
11-9198,"Personal Service Managers; Entertainment & Recreation Managers, Except Gambling; and Managers, All Other",NaN,NaN,24520.0,16900.0
13-1198,"Project Management Specialists and Business Operations Specialists, All Other",NaN,NaN,35770.0,NaN
13-2098,"Financial and Investment Analysts, Financial Risk Specialists, and Financial Specialists, All Other",NaN,NaN,52610.0,590.0
15-1245,Database Administrators and Architects,NaN,NaN,5300.0,NaN
15-1256,Software Developers and Software Quality Assurance Analysts and Testers,NaN,NaN,61640.0,600.0
15-1257,Web Developers and Digital Interface Designers,NaN,NaN,9460.0,730.0
15-2098,"Data Scientists and Mathematical Science Occupations, All Other",NaN,NaN,2910.0,NaN
17-3098,"Calibration and Engineering Technologists and Technicians, Except Drafters, All Other",NaN,NaN,390.0,NaN
21-1018,"Substance Abuse, Behavioral Disorder, and Mental Health Counselors",NaN,6970.0,8490.0,9200.0


**Code findings**
- 2016-26 and 2018-28 use SOC 2010. 2020-30 and 2022-32 use SOC 2018. A trend across editions needs the SOC 2010→2018 crosswalk (notebook 06) before any comparison.
- Codes ending in `8` (e.g. `15-1256`, `13-1198`, `29-2098`) aren't in the SOC lists at all. They're BLS Employment Projections aggregates that combine detailed SOC codes OEWS can't separate. They need their own mapping rows in the job dimension, or they won't join to OEWS, O\*NET or CIP→SOC.
- The apostrophe prefix is in the two Wayback files only.

## 5. Coverage

One region (NYC), four editions, 2016 → 2032. The base year matters: 2020-30 starts in the COVID trough.

In [11]:
tot = nyc[nyc.SOC == "00-0000"].set_index("edition")[["Base Year_n", "Projected Year_n", "Change_n", "Percent_n",
                                                      "Total Annual Openings_n", "Growth Annual Openings_n", "Replacement Annual Openings_n"]]
tot.columns = [c.replace("_n", "") for c in tot.columns]
display(tot)

mg = nyc[nyc.level == "major"].pivot_table(index=["SOC", "Occupation Title"], columns="edition", values="Percent_n", aggfunc="first")
print("10-year % change by major group (NYC):")
mg.sort_values("2020-2030", ascending=False).head(8)

,Base Year,Projected Year,Change,Percent,Total Annual Openings,Growth Annual Openings,Replacement Annual Openings
edition,,,,,,,
2016-2026,4684200,5325230,641030,13.70,593100,64100,529000
2018-2028,4939450,5543240,603790,12.20,641095,60379,580720
2020-2030,4566620,5704650,1138030,24.90,653913,113803,540110
2022-2032,4850820,5435330,584510,12.05,605681,58451,547230


10-year % change by major group (NYC):


,edition,2016-2026,2018-2028,2020-2030,2022-2032
SOC,Occupation Title,,,,
35-0000,Food Preparation and Serving Related Occupations,20.6,4.2,81.6,17.77
39-0000,Personal Care and Service Occupations,26.8,35.7,54.1,15.59
31-0000,Healthcare Support Occupations,42.6,55.6,35.7,32.25
53-0000,Transportation and Material Moving Occupations,9.5,9.3,32.4,6.34
21-0000,Community and Social Service Occupations,19.8,22.9,29.4,22.36
15-0000,Computer and Mathematical Occupations,21.5,19.5,24.8,10.54
27-0000,"Arts, Design, Entertainment, Sports, and Media Occupations",7.9,4.9,24.8,4.46
37-0000,Building and Grounds Cleaning and Maintenance Occupations,16.2,10.4,24.6,12.56


**Coverage findings**
- 2020-30 is distorted by its base year. It projects +24.9% for NYC overall and +81.6% for Food Preparation and Serving. The other editions sit around 12–14%. That's recovery from a 2020 base measured during lockdown, not structural growth. 2020-30 can't be read as a trend point without a caveat, and it shouldn't drive "growing jobs" rankings.
- Base employment: 4.68M (2016) → 4.94M (2018) → 4.57M (2020) → 4.85M (2022).
- The occupation lists change between editions (codes added and dropped, partly from SOC changes, partly from suppression; see §7).

## 6. Relationships: will it join?

The target join is SOC → the job dimension, which other sources feed: OEWS wages, BLS education requirements, O\*NET, and the CIP→SOC bridge. The NYS DOL wage table (`nysdol_oews_regions`, OEWS 2025, NYC) is the closest sibling, so the check is the share of current-edition NYC leaf codes that find an NYC wage row.

In [12]:
w = pd.read_csv(RAW / "jobs/nysdol_oews_regions/gkgz-nw24_2026-10-12.csv", dtype=str, keep_default_na=False)
w.columns = [c.strip() for c in w.columns]
w_nyc = w[w["Area Name"].str.strip() == "New York City"].copy()
w_codes = set(w_nyc["Standard Occupational Code"].str.strip())

cur = nyc[(nyc.edition == "2022-2032") & nyc.leaf].copy()
cur["in_wage_table"] = cur.SOC.isin(w_codes)
print(f"2022-32 NYC leaves: {len(cur)} | matched to NYC wage table: {cur.in_wage_table.sum()} ({cur.in_wage_table.mean():.0%})")
print(f"Share of leaf base employment matched: {cur.loc[cur.in_wage_table, 'Base Year_n'].sum() / cur['Base Year_n'].sum():.0%}")
cur[~cur.in_wage_table].sort_values("Base Year_n", ascending=False)[["SOC", "Occupation Title", "level", "Base Year_n"]].head(12)

2022-32 NYC leaves: 543 | matched to NYC wage table: 508 (94%)
Share of leaf base employment matched: 98%


,SOC,Occupation Title,level,Base Year_n
1949,11-9198,"Personal Service Managers; Entertainment & Recreation Managers, Ex...",detailed,16900
2553,53-4031,Railroad Conductors and Yardmasters,detailed,5400
2554,53-4041,Subway and Streetcar Operators,detailed,5070
2549,53-3058,"Passenger Vehicle Drivers, Except Bus Drivers, Transit and Intercity",detailed,4660
2152,27-2011,Actors,detailed,4540
2158,27-2042,Musicians and Singers,detailed,4520
2457,49-2095,"Electrical and Electronics Repairers, Powerhouse, Substation, and ...",detailed,2110
2299,39-1098,First-Line Supervisors of Personal Service & Entertainment and Rec...,detailed,1860
2108,25-1199,"Postsecondary Teachers, All Other",detailed,1810
2350,41-9091,"Door-to-Door Sales Workers, News and Street Vendors, and Related W...",detailed,1810


**Join findings.** 94% of 2022-32 leaves (98% of their employment) join to the NYC wage table on SOC as-is. The misses are of two kinds: BLS projection aggregates (`11-9198`, `39-1098`, `53-3058` …), which need a projection-code → SOC 2018 bridge (one-to-many, notebook 06), and occupations the wage table doesn't publish for NYC (railroad conductors, actors, musicians). Those simply won't have wages.

## 7. Tie-out

No published NYC total we can fetch: the portal page lists no regional totals, and the NYS DOL Tableau dashboard isn't machine-readable. Open item: read the NYC total off the dashboard by hand. Until then, three checks.

**(a) Internal arithmetic.** Under the separations method, `Total openings = Growth + Replacement` and `Growth openings = Change / 10`.

In [13]:
n = nyc
chk = pd.DataFrame({
    "edition": n.edition,
    "open != growth+repl (>1)": (n["Total Annual Openings_n"] - n["Growth Annual Openings_n"] - n["Replacement Annual Openings_n"]).abs() > 1,
    "max |open - growth - repl|": (n["Total Annual Openings_n"] - n["Growth Annual Openings_n"] - n["Replacement Annual Openings_n"]).abs(),
    "growth != change/10 (>1)": (n["Growth Annual Openings_n"] - n["Change_n"] / 10).abs() > 1,
    "percent != change/base (>0.2pt)": (n["Percent_n"] - n["Change_n"] / n["Base Year_n"] * 100).abs() > 0.2,
    "projected != base+change": n["Projected Year_n"] != n["Base Year_n"] + n["Change_n"],
})
chk.groupby("edition").agg({c: ("max" if c.startswith("max") else "sum") for c in chk.columns if c != "edition"})

,open != growth+repl (>1),max |open - growth - repl|,growth != change/10 (>1),percent != change/base (>0.2pt),projected != base+change
edition,,,,,
2016-2026,223,10,461,0,0
2018-2028,451,11,0,0,0
2020-2030,0,0,0,0,0
2022-2032,0,0,0,0,0


- **2020-30 and 2022-32 are exact.**
- **2018-28** is off by at most a few openings. The components are rounded to 10s and the total isn't, so this is a rounding artifact.
- **2016-26** has growth openings that don't equal change/10 (±1–4) and totals off by ±10. Everything in that edition is rounded to 10s. It's tolerable, but the 2016-26 openings shouldn't be compared to later editions at single-digit precision.

(b) Do the leaves add up to the total?

In [14]:
cov = []
for ed, g in nyc.groupby("edition"):
    t = g[g.SOC == "00-0000"].iloc[0]
    L = g[g.leaf]
    cov.append({"edition": ed, "total jobs": t["Base Year_n"], "sum of leaves": L["Base Year_n"].sum(),
                "jobs unassigned": t["Base Year_n"] - L["Base Year_n"].sum(),
                "leaf coverage": L["Base Year_n"].sum() / t["Base Year_n"],
                "openings coverage": L["Total Annual Openings_n"].sum() / t["Total Annual Openings_n"]})
display(pd.DataFrame(cov).set_index("edition").style.format({"leaf coverage": "{:.1%}", "openings coverage": "{:.1%}",
                                                            "total jobs": "{:,.0f}", "sum of leaves": "{:,.0f}", "jobs unassigned": "{:,.0f}"}))

# Where the 2022-32 gap sits: major group total vs sum of its leaves
g = nyc[nyc.edition == "2022-2032"]
maj = g[g.level == "major"].set_index(g[g.level == "major"].SOC.str[:2])
leafsum = g[g.leaf].groupby(g[g.leaf].SOC.str[:2])["Base Year_n"].sum()
gap = pd.DataFrame({"major group": maj["Occupation Title"], "group total": maj["Base Year_n"], "sum of leaves": leafsum})
gap["unassigned"] = gap["group total"] - gap["sum of leaves"]
gap["coverage"] = gap["sum of leaves"] / gap["group total"]
gap.sort_values("unassigned", ascending=False).head(8).style.format({"coverage": "{:.0%}", "group total": "{:,.0f}", "sum of leaves": "{:,.0f}", "unassigned": "{:,.0f}"})

,total jobs,sum of leaves,jobs unassigned,leaf coverage,openings coverage
edition,,,,,
2016-2026,"4,684,200","4,620,950","63,250",98.6%,98.9%
2018-2028,"4,939,450","4,840,230","99,220",98.0%,98.2%
2020-2030,"4,566,620","4,436,400","130,220",97.1%,97.9%
2022-2032,"4,850,820","4,472,160","378,660",92.2%,94.2%


,major group,group total,sum of leaves,unassigned,coverage
SOC,,,,,
13,Business and Financial Operations Occupations,"460,440","349,280","111,160",76%
15,Computer and Mathematical Occupations,"181,080","88,970","92,110",49%
11,Management Occupations,"370,110","320,000","50,110",86%
29,Healthcare Practitioners and Technical Occupations,"245,520","210,890","34,630",86%
53,Transportation and Material Moving Occupations,"272,710","246,690","26,020",90%
33,Protective Service Occupations,"182,170","167,060","15,110",92%
25,Educational Instruction and Library Occupations,"306,970","294,460","12,510",96%
19,"Life, Physical, and Social Science Occupations","28,970","22,550","6,420",78%


The 2016-26 through 2020-30 editions assign 97–99% of jobs to a leaf, which is normal suppression of small or confidential cells. 2022-32 assigns only 92%: 379k NYC jobs belong to no published occupation. Half the computer & math group and a quarter of business & financial are missing. That's not ordinary suppression, so trace it:

(c) Trace one record: Software Developers, against the NYS DOL wage table (OEWS 2025)

In [15]:
sw = nyc[nyc.SOC.isin(["15-1200", "15-1256", "15-1257"])].pivot_table(index=["SOC", "Occupation Title"], columns="edition", values="Base Year_n", aggfunc="first")
display(sw)
print("NYS DOL wage table, NYC (OEWS 2025):")
w_nyc[w_nyc["Standard Occupational Code"].str.strip().str.startswith("15-125")][["Standard Occupational Code", "Occupational Title", "Employment"]]

,edition,2020-2030,2022-2032
SOC,Occupation Title,,
15-1200,Computer Occupations,163070,166460
15-1256,Software Developers and Software Quality Assurance Analysts and Testers,61640,600
15-1257,Web Developers and Digital Interface Designers,9460,730


NYS DOL wage table, NYC (OEWS 2025):


,Standard Occupational Code,Occupational Title,Employment
4662,15-1251,Computer Programmers,2900
4663,15-1252,Software Developers,74710
4664,15-1253,Software Quality Assurance Analysts and Testers,6100
4665,15-1254,Web Developers,2220
4666,15-1255,Web and Digital Interface Designers,11110


In [16]:
# Same occupation in every region of the current file: is the problem NYC-only?
d = raw[FILES.file[FILES.note.str.contains("current")].iloc[0]]
d[d.SOC.isin(["15-1252", "15-1256"])].pivot_table(index="Area", columns="SOC", values="Base Year", aggfunc="first")

SOC,15-1252,15-1256
Area,,
Capital Region,NaN,110
Finger Lakes,NaN,130
Hudson Valley,NaN,160
Long Island,NaN,210
New York City,NaN,600
New York Statewide,108010,NaN
North Country,NaN,30
Western Region,NaN,130


This is a publisher defect, not a real number, and it has a pattern. In the 2022-32 NYC edition, `15-1256 Software Developers and QA Testers` shows 600 jobs with 0 replacement openings. The same code showed 61,640 in 2020-30, and the NYS DOL wage table counts 74,710 NYC software developers plus 6,100 QA testers. Every region has implausibly tiny values for it (Long Island 210, Capital Region 110), while the statewide 2024-34 block reports 108k under the detailed code `15-1252`. §4's table shows the same collapse in other BLS projection-aggregate codes: `13-2098 Financial and Investment Analysts…` falls from 52,610 → 590 and `15-1257 Web Developers…` from 9,460 → 730. `13-1198 Project Management Specialists` (35,770) and `15-2098 Data Scientists` are gone entirely. The regional 2022-32 extract seems to have broken wherever an aggregate code is involved. In total, 26 leaves worth ~98k (2020) jobs vanish between editions.

For a CUNY project this is the worst place for a hole. The broken codes are exactly the jobs computer science, finance, business and data majors target.

**Checked against the official workbook.** The NYS DOL regional workbook ([`statistics-regional-long-term-occupational-projections`](https://dol.ny.gov/statistics-regional-long-term-occupational-projections), xlsx, sheet *New York City*) shows the same values: `15-1256` = 600, `13-2098` = 590, `15-1257` = 730, while `15-0000` Computer and Mathematical = 181,080. So the hole is in NYS DOL's publication itself, not in the data.ny.gov extract. No published version has correct detail for these codes.


In [17]:
cur_codes = set(nyc[nyc.edition == "2022-2032"].SOC)
prev = nyc[(nyc.edition == "2020-2030") & nyc.leaf & ~nyc.SOC.isin(cur_codes)]
print(f"Leaves in 2020-30 NYC that vanish from 2022-32: {len(prev)} codes, {prev['Base Year_n'].sum():,.0f} jobs (2020 base)")
prev.sort_values("Base Year_n", ascending=False)[["SOC", "Occupation Title", "Base Year_n"]].head(10)

Leaves in 2020-30 NYC that vanish from 2022-32: 26 codes, 98,400 jobs (2020 base)


,SOC,Occupation Title,Base Year_n
1342,13-1198,Project Management Specialists and Business Operations Specialists...,35770
1302,11-3010,Administrative Services and Facilities Managers,11250
1581,29-2040,Emergency Medical Technicians and Paramedics,8660
1590,29-2098,"Medical Dosimetrists, Medical Records Specialists, and Health Tech...",6000
1300,11-2030,Public Relations and Fundraising Managers,5990
1617,33-1090,"Miscellaneous First-Line Supervisors, Protective Service Workers",5800
1363,15-1245,Database Administrators and Architects,5300
1630,33-9098,"School Bus Monitors and Protective Service Workers, All Other",3930
1660,37-3019,"Grounds Maintenance Workers, All Other",3260
1372,15-2098,"Data Scientists and Mathematical Science Occupations, All Other",2910


## 8. Fitness for purpose

| Question | Fit | Why |
|---|---|---|
| Q1: which NYC jobs grow or shrink, and how many openings a year | Use with limits | It's the only NYC-specific projection, and the method is sound (BLS separations). But the current edition (2022-32) has a hole of ~379k jobs, including Software Developers, and 2020-30 is distorted by its COVID base year |
| Openings vs graduates (the gap) | partial | Annual openings include separations (exits and transfers), so openings ≠ net new jobs. That's the right measure for "how many jobs will graduates compete for", but it's a 10-year average |
| Trend across editions | partial | Needs SOC 2010→2018 and the projection-aggregate bridge first. Treat 2020-30 as an outlier |
| Anything below NYC (borough) | no | Not published |
| Who fills the openings (education) | no | Not in this source. Comes from BLS education (notebook 04) |

The 2022-32 hole: options and decision (see `docs/decisions.md`)
- **A)** Find corrected NYC figures from NYS DOL. Not possible. *Checked:* the official xlsx has the same defect (§7c). An email to NYS DOL is still worth sending; a corrected file would replace the fill below.
- **B)** Copy the broken codes from 2020-30. Rejected: brings in the COVID-trough base year for exactly these jobs.
- **C)** Trust 2022-32 only at group level, where totals are complete. Part of the decision.
- **D)** Use BLS national projections × NYC OEWS staffing to fill detail. Part of the decision.

**Going with C + D.** Keep the published 2022-32 group totals. Within each group, the unassigned residual (group total − sum of its published leaves) is split across the broken or missing codes: base employment in proportion to their NYC OEWS 2025 employment, projected employment and openings weighted by BLS national 2024-34 growth and openings rates, then rescaled so the group still adds up. Those rows get `quality_flag = 'imputed'`. This needs joins (OEWS, BLS EP), so it's built in core, not staging.


## 9. Issues list and proposed staging table

| # | Issue | Rows affected (NYC) | Proposed treatment (decided in staging) |
|---|---|---|---|
| 1 | Each file mixes 2 editions (regions = N, statewide = N+2) | all files | Take the edition from `Period` on each row; never from the file name |
| 2 | `Period` format varies (`2020-2030` vs `2020 - 2030`) | 2020-30 regions | Strip spaces → `edition`; derive `base_year`, `projected_year` |
| 3 | Leading apostrophe on SOC | 2016-26, 2018-28 (all rows) | `lstrip("'")` |
| 4 | Exact duplicate row (`39-7000`) | 1 (2018-28) | Drop exact duplicates; assert key unique after |
| 5 | Blank rows and conflicting duplicates outside NYC | 0 NYC (2020-30, 2022-32 other areas) | Out of scope once filtered to NYC; assert the NYC key is unique |
| 6 | `Base Year` / `Projected Year` hold employment, not years | all | Rename to `base_employment` / `projected_employment` |
| 7 | All SOC levels stacked; hierarchy is ragged | all | Add `soc_level` and `is_leaf` (computed per edition). Facts are summed over leaves only |
| 8 | SOC 2010 (2016-26, 2018-28) vs SOC 2018 (2020-30, 2022-32) | 2 editions | Keep the original code + `soc_edition`; map to 2018 in core via the crosswalk (nb 06) |
| 9 | Projection aggregate codes (`…8`, e.g. 15-1256) not in SOC | ~10–15 per edition | Bridge table projection-code → SOC 2018 detailed codes (nb 06) |
| 10 | 2022-32 aggregate-code defect: Software Devs 600 (vs ~75k OEWS), Financial Analysts 590 (vs 52,610 in 2020-30); ~379k jobs (7.8%) unassigned | 2022-32 | **Decided: C + D** (§8). Staging keeps the published values with `quality_flag = 'publisher_defect'`; core replaces them with the group-residual fill, `quality_flag = 'imputed'` |
| 11 | 2020-30 base year = COVID trough (+24.9% total, +81.6% food prep) | 2020-30 | Keep, but flag the edition as `covid_base`; exclude it from "fastest-growing" rankings by default |
| 12 | 2016-26 values rounded to 10s; identities off by ≤10 | 2016-26 | Keep as published; document the precision |
| 13 | Annual openings are a 10-year average | all | Name the column `annual_openings_avg`; say so in the dashboard tooltip |

**Proposed staging table:** `stg_nysdol__occupation_projections`
- **Grain / key:** one row per `edition × soc_code` (NYC only). Key `(edition, soc_code)`, asserted unique.
- **Columns:** `edition` text (`2022-2032`), `base_year` int, `projected_year` int, `soc_code` text, `soc_edition` text (`2010`/`2018`), `soc_title` text, `soc_level` text, `is_leaf` bool, `base_employment` int, `projected_employment` int, `employment_change` int, `pct_change` numeric, `annual_openings_avg` int, `growth_openings_avg` int, `separation_openings_avg` int, `quality_flag` text, `source_file` text.
- **Filter:** `Area = 'New York City'`. Rows in → out: 657 + 655 + 638 + 626 = 2,576 NYC rows in → 2,575 out (−1 exact duplicate).

In [18]:
# Row accounting for the proposed staging filter, recomputed rather than typed
rows_in = sum((d.Area == "New York City").sum() for d in raw.values())
print(f"NYC rows in: {rows_in:,} -> after exact-duplicate removal: {len(nyc):,} (dropped {rows_in - len(nyc)})")
assert not nyc.duplicated(["edition", "SOC"]).any()

NYC rows in: 2,576 -> after exact-duplicate removal: 2,575 (dropped 1)
